In [1]:
%pylab inline

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


In [9]:
from pycbc.events import hm_utils

In [154]:
from numba import njit

In [221]:
a_len = 5120
t2_coinc_window = 8
t3_coinc_window = 17
t23_coinc_window = 17
det_idx = hm_utils.index_combinations(a_len, t2_coinc_window, 
    t3_coinc_window, np.int64)

In [222]:
# generate some random data
random.seed(1234)
_ii = sort(random.choice(arange(len(det_idx)), 5000))
snrs = random.chisquare(2, len(_ii))
det_idx = det_idx[_ii]

In [229]:
a = array([1,2,3])
b = a[array([True, False, True])]

In [231]:
b[0] = 9

In [232]:
a, b

(array([1, 2, 3]), array([9, 3]))

In [243]:
@njit
def _check_time_idx_is_sorted_bool(time_idx):
    cond = [time_idx[i] <= time_idx[i+1] 
        for i in range(len(time_idx)-1)]
    return cond

def check_time_idx_is_sorted(time_idx):
    cond = _check_time_idx_is_sorted_bool(time_idx)
    assert np.all(cond), "time_idx must be sorted"
    
def maximal_coinc_in_ifo(snrs, time_idx, check_sorted=True):
    """Choose the maximum network snr for each time point.
    Requires time_idx to be sorted.
    Parameters
    ----------
    snrs: numpy.array
        SNR-like values to maximize over.
    time_idx: numpy.array
        Sorted time indices.
    Returns
    -------
    i_max: numpy.array
        The indices that maximize the snr.
    """
    if check_sorted: check_time_idx_is_sorted(time_idx)
    i_max = []
    j_start, j_end = 0, 0
    for c in np.unique(time_idx, return_counts=True)[1]:
        j_end += c
        i_max.append(np.argmax(snrs[j_start:j_end]) + j_start)
        j_start += c
    i_max = np.array(i_max)
    return i_max

def maximize_snr_per_timepoint(snrs, det_idx, check_sorted=True):
    """Choose the maximum network snr for each time point in each detector.
    Assumes three detector network.
    Parameters
    ----------
    snrs: numpy.array
        SNR-like values to maximize over.
    det_idx: numpy.array2d
        Time indices. Must be sorted in zeroth detector.
    Returns
    -------
    snrs: numpy.array
        The maximum SNR-like values.
    det_idx: numpy.array2d
        Maximum SNR time indices. Sorted in zeroth detector time.
    i_max: numpy.array
        The indices that maximize the snr.
    """
    i_max = {0: maximal_coinc_in_ifo(snrs, det_idx[:,0], check_sorted=True)}
    snrs = snrs[i_max[0]]
    det_idx = det_idx[i_max[0]]
#     # check if sorted after first step as more efficient and 
#     # will likely still catch it.
#     if check_was_sorted: check_time_idx_is_sorted(det_idx[:,0])
    nifos = shape(det_idx)[-1]
    for i in range(1, nifos):
        sort_idx = np.argsort(det_idx[:,i])
        snrs = snrs[sort_idx]
        det_idx = det_idx[sort_idx]
        idx = maximal_coinc_in_ifo(snrs, det_idx[:,i], check_sorted=False)
        i_max[i] = i_max[i-1][sort_idx][idx]
        snrs = snrs[idx]
        det_idx = det_idx[idx]
    # sort so that ifo 0 times are in order.
    sort_idx = np.argsort(det_idx[:,0])
    snrs, det_idx, i_max = snrs[sort_idx], det_idx[sort_idx], i_max[2][sort_idx]
    return snrs, det_idx, i_max

DECIDED:

* return just the indices somehow. 

In [226]:
%timeit maximize_snr_per_timepoint(snrs, det_idx, True)

18.9 ms ± 84.1 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [227]:
%timeit maximize_snr_per_timepoint(snrs, det_idx, False)

18.8 ms ± 32.9 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [193]:
snrs, det_idx, i_max = maximize_snr_per_timepoint(snrs, det_idx)
print(len(det_idx))

2455


In [196]:
print(len(det_idx))
idx = maximal_coinc_in_ifo(snrs, det_idx[:,0])
det_idx = det_idx[idx]
snrs = snrs[idx]
for i in [1,2]:
    sort_idx = np.argsort(det_idx[:,i])
    idx = maximal_coinc_in_ifo(snrs[sort_idx], det_idx[:,i][sort_idx])
    det_idx = det_idx[sort_idx][idx]
    snrs = snrs[sort_idx][idx]
    len(det_idx), len(idx)
print(len(det_idx))

3039734
2455
